# Aula 03 — MLP, camadas densas e convenções de shape

Este tutorial monta uma rede densa de duas camadas **somente com NumPy**. O foco é o
contrato estrutural: eixos, shapes, parâmetros, broadcasting e invariantes do forward.
Também construiremos uma solução explícita para XOR, sem treinamento e sem autograd.

**Dependências mínimas:** Python 3.11, NumPy 1.26, Matplotlib 3.8 e nbformat 5.9.
Os dados são pequenos, explícitos ou gerados por seed fixa; não há download nem segredo.


## Goal

Ao terminar, você deverá conseguir:

1. traduzir uma arquitetura $d_0\to d_1\to d_2$ em matrizes compatíveis;
2. explicar por que exemplos ficam nas linhas e unidades nas colunas;
3. provar que uma camada densa vetorizada equivale a vários neurônios individuais;
4. contar pesos e vieses sem incluir o tamanho do lote;
5. testar invariância à permutação do lote e independência entre exemplos;
6. detectar um broadcasting que preserva o shape, mas viola a semântica;
7. mostrar como uma camada oculta cria uma representação que separa XOR.

Ativações serão derivadas na Aula 04; cache do forward, losses e backward virão depois.


## Setup

Adotaremos uma única convenção em todo o M5:

- $X\in\mathbb{R}^{n\times d_0}$: exemplos nas linhas, atributos nas colunas;
- $W^{[\ell]}\in\mathbb{R}^{d_{\ell-1}\times d_\ell}$: uma unidade por coluna;
- $b^{[\ell]}\in\mathbb{R}^{1\times d_\ell}$: uma linha compartilhada pelo lote;
- $Z^{[\ell]},A^{[\ell]}\in\mathbb{R}^{n\times d_\ell}$;
- `SEED = 20260909`; `float64` para tornar as verificações numéricas rigorosas.


In [ ]:
from __future__ import annotations

import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

SEED = 20260909
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=6, suppress=True)

print({
    "python": platform.python_version(),
    "numpy": np.__version__,
    "matplotlib": matplotlib.__version__,
    "seed": SEED,
})


## Steps

### 1. O contrato de uma camada densa

Uma camada com `in_features = d_in` e `out_features = d_out` calcula
$Z=XW+b$. A implementação abaixo exige explicitamente `b.shape == (1, d_out)`.
O NumPy aceitaria um vetor `(d_out,)`, mas a forma bidimensional deixa o eixo de
unidades visível e evita ambiguidades didáticas.


In [ ]:
def dense(X: np.ndarray, W: np.ndarray, b: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=np.float64)
    W = np.asarray(W, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    assert X.ndim == 2, "X deve ser (lote, atributos)"
    assert W.ndim == 2, "W deve ser (entradas, unidades)"
    assert b.ndim == 2 and b.shape[0] == 1, "b deve ser (1, unidades)"
    assert X.shape[1] == W.shape[0], "atributos de X devem casar com linhas de W"
    assert W.shape[1] == b.shape[1], "colunas de W devem casar com unidades de b"
    Z = X @ W + b
    assert Z.shape == (X.shape[0], W.shape[1])
    assert np.isfinite(Z).all()
    return Z


def relu(Z: np.ndarray) -> np.ndarray:
    return np.maximum(Z, 0.0)


def step(Z: np.ndarray) -> np.ndarray:
    return (Z >= 0.0).astype(int)


### 2. Exemplo resolvido à mão

Duas observações com três atributos alimentam duas unidades ocultas. Depois da ReLU,
uma unidade de saída produz um score por observação. Os valores esperados foram
calculados antes da execução e funcionam como oráculo independente.


In [ ]:
X_hand = np.array([[1.0, 0.0, -1.0], [2.0, 1.0, 0.0]])
W1_hand = np.array([[1.0, -1.0], [2.0, 0.5], [-1.0, 1.0]])
b1_hand = np.array([[0.5, -0.5]])
W2_hand = np.array([[2.0], [-3.0]])
b2_hand = np.array([[0.25]])

Z1_hand = dense(X_hand, W1_hand, b1_hand)
A1_hand = relu(Z1_hand)
Z2_hand = dense(A1_hand, W2_hand, b2_hand)

expected_Z1 = np.array([[2.5, -2.5], [4.5, -2.0]])
expected_Z2 = np.array([[5.25], [9.25]])
assert np.array_equal(Z1_hand, expected_Z1)
assert np.array_equal(Z2_hand, expected_Z2)
print("Z[1] =\n", Z1_hand)
print("A[1] =\n", A1_hand)
print("Z[2] =\n", Z2_hand)


### 3. Matriz versus neurônios individuais

Na nossa convenção, a coluna `W[:, j]` contém todos os pesos da unidade $j$.
Logo, `X @ W + b` deve ser idêntico a calcular `X @ W[:, j] + b[0, j]`
para cada unidade e empilhar os resultados.


In [ ]:
X_equiv = rng.normal(size=(7, 5))
W_equiv = rng.normal(size=(5, 4))
b_equiv = rng.normal(size=(1, 4))

Z_matrix = dense(X_equiv, W_equiv, b_equiv)
Z_neurons = np.column_stack([
    X_equiv @ W_equiv[:, j] + b_equiv[0, j]
    for j in range(W_equiv.shape[1])
])
equivalence_error = float(np.max(np.abs(Z_matrix - Z_neurons)))
assert equivalence_error < 1e-14
print({"erro_maximo_matriz_vs_neuronios": equivalence_error})


### 4. Arquitetura de duas camadas e rastreamento de shapes

`forward_two_layers` devolve apenas um **traço de inspeção**. Não é o cache de treino
que será projetado na Aula 05. As ativações recebem funções explícitas para deixar clara
a fronteira entre transformação afim e não linearidade.


In [ ]:
def forward_two_layers(
    X: np.ndarray,
    W1: np.ndarray,
    b1: np.ndarray,
    W2: np.ndarray,
    b2: np.ndarray,
    *,
    hidden_activation=relu,
    output_activation=lambda z: z,
):
    Z1 = dense(X, W1, b1)
    A1 = hidden_activation(Z1)
    assert A1.shape == Z1.shape
    Z2 = dense(A1, W2, b2)
    A2 = output_activation(Z2)
    assert A2.shape == Z2.shape
    return A2, {
        "X": X.shape,
        "W1": W1.shape,
        "b1": b1.shape,
        "Z1": Z1.shape,
        "A1": A1.shape,
        "W2": W2.shape,
        "b2": b2.shape,
        "Z2": Z2.shape,
        "A2": A2.shape,
    }


architecture = (3, 4, 2)
W1 = rng.normal(size=(architecture[0], architecture[1]))
b1 = rng.normal(size=(1, architecture[1]))
W2 = rng.normal(size=(architecture[1], architecture[2]))
b2 = rng.normal(size=(1, architecture[2]))
X = rng.normal(size=(6, architecture[0]))

output, shape_trace = forward_two_layers(X, W1, b1, W2, b2)
for name, shape in shape_trace.items():
    print(f"{name:>3}: {shape}")
assert output.shape == (6, 2)


### 5. Contagem independente de parâmetros

Para $d_0\to d_1\to d_2$, a contagem é
$(d_0+1)d_1+(d_1+1)d_2$. O `+1` incorpora um viés por unidade.
O lote não aparece: aumentar $n$ muda ativações e custo de execução, não o modelo.


In [ ]:
def parameter_count(widths: tuple[int, ...]) -> int:
    assert len(widths) >= 2 and all(width > 0 for width in widths)
    return sum((fan_in + 1) * fan_out for fan_in, fan_out in zip(widths[:-1], widths[1:]))


formula_count = parameter_count(architecture)
array_count = W1.size + b1.size + W2.size + b2.size
assert formula_count == array_count == 26

for batch_size in (1, 7, 32):
    X_batch = rng.normal(size=(batch_size, architecture[0]))
    y_batch, trace = forward_two_layers(X_batch, W1, b1, W2, b2)
    assert y_batch.shape == (batch_size, architecture[2])
    print({
        "lote": batch_size,
        "parametros": formula_count,
        "elementos_ativacoes_X_Z1_Z2": batch_size * sum(architecture),
        "saida": y_batch.shape,
    })


### 6. Propriedades do eixo do lote

Uma MLP densa sem operação entre exemplos aplica a mesma função a cada linha.
Portanto, permutar o lote deve apenas permutar as saídas; acrescentar outras linhas não
pode mudar a saída de uma linha existente. Esses testes capturam erros que um teste de
shape isolado não encontra.


In [ ]:
X_property = rng.normal(size=(11, architecture[0]))
base_output, _ = forward_two_layers(X_property, W1, b1, W2, b2)
permutation = rng.permutation(len(X_property))
permuted_output, _ = forward_two_layers(X_property[permutation], W1, b1, W2, b2)
permutation_error = float(np.max(np.abs(permuted_output - base_output[permutation])))

first_alone, _ = forward_two_layers(X_property[:1], W1, b1, W2, b2)
first_in_batch = base_output[:1]
batch_independence_error = float(np.max(np.abs(first_alone - first_in_batch)))

duplicated = np.vstack([X_property[:1], X_property[:1]])
duplicate_output, _ = forward_two_layers(duplicated, W1, b1, W2, b2)
duplicate_error = float(np.max(np.abs(duplicate_output[0] - duplicate_output[1])))

assert permutation_error < 1e-14
assert batch_independence_error < 1e-14
assert duplicate_error < 1e-14
print({
    "erro_permutacao": permutation_error,
    "erro_independencia_lote": batch_independence_error,
    "erro_linhas_duplicadas": duplicate_error,
})


### 7. Armadilha: broadcasting com shape correto e significado errado

Se `n == d_out == 3`, um viés `(3, 1)` soma um valor diferente a cada **exemplo**.
O resultado ainda tem shape `(3, 3)`, mas deixou de representar um viés por unidade.
Ao permutar as entradas sem permutar esse falso viés, a equivariância do lote quebra.


In [ ]:
X_trap = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
W_trap = np.array([[1.0, 2.0, -1.0], [0.5, -1.0, 3.0]])
b_correct = np.array([[0.1, -0.2, 0.3]])
b_wrong = np.array([[0.1], [-0.2], [0.3]])

Z_correct = X_trap @ W_trap + b_correct
Z_wrong = X_trap @ W_trap + b_wrong
assert Z_correct.shape == Z_wrong.shape == (3, 3)

p = np.array([2, 0, 1])
correct_permuted = X_trap[p] @ W_trap + b_correct
wrong_permuted = X_trap[p] @ W_trap + b_wrong
correct_contract_error = float(np.max(np.abs(correct_permuted - Z_correct[p])))
wrong_contract_error = float(np.max(np.abs(wrong_permuted - Z_wrong[p])))

strict_rejected = False
try:
    dense(X_trap, W_trap, b_wrong)
except AssertionError:
    strict_rejected = True

assert correct_contract_error == 0.0
assert wrong_contract_error > 0.0
assert strict_rejected
print({
    "shape_enganoso": Z_wrong.shape,
    "erro_contrato_correto": correct_contract_error,
    "erro_contrato_errado": wrong_contract_error,
    "dense_estrita_rejeitou": strict_rejected,
})


### 8. XOR com uma representação oculta

Uma unidade limiar não separa XOR. Duas unidades ocultas podem codificar `OR` e `AND`:
$h_1=\mathbb{1}[x_1+x_2-0{,}5\ge0]$ e
$h_2=\mathbb{1}[x_1+x_2-1{,}5\ge0]$.
Na representação $(h_1,h_2)$, a saída
$\mathbb{1}[h_1-2h_2-0{,}5\ge0]$ separa os casos positivos.


In [ ]:
X_xor = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_xor = np.array([0, 1, 1, 0])

W1_xor = np.array([[1.0, 1.0], [1.0, 1.0]])
b1_xor = np.array([[-0.5, -1.5]])
W2_xor = np.array([[1.0], [-2.0]])
b2_xor = np.array([[-0.5]])

H_xor = step(dense(X_xor, W1_xor, b1_xor))
y_hat_xor = step(dense(H_xor, W2_xor, b2_xor)).ravel()

assert np.array_equal(H_xor, np.array([[0, 0], [1, 0], [1, 0], [1, 1]]))
assert np.array_equal(y_hat_xor, y_xor)
print("[x1, x2] -> [OR, AND] -> XOR")
for x_i, h_i, y_i in zip(X_xor.astype(int), H_xor, y_hat_xor):
    print(f"{x_i.tolist()} -> {h_i.tolist()} -> {int(y_i)}")
print({"acuracia_xor": float(np.mean(y_hat_xor == y_xor)), "parametros": parameter_count((2, 2, 1))})


### 9. Visualização da mudança de representação

O painel esquerdo mostra o padrão diagonalmente alternado no espaço de entrada.
O direito mostra que as unidades ocultas fundem os dois positivos em $(1,0)$ e mantêm
os negativos em posições separáveis por uma reta. A sobreposição é intencional e está
anotada com multiplicidade.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
colors = np.where(y_xor == 1, "tab:orange", "tab:blue")

axes[0].scatter(X_xor[:, 0], X_xor[:, 1], c=colors, s=110, edgecolor="black")
for x_i, y_i in zip(X_xor, y_xor):
    axes[0].annotate(f"y={y_i}", x_i + np.array([0.03, 0.03]))
axes[0].set(title="Espaço de entrada: XOR", xlabel="$x_1$", ylabel="$x_2$")

axes[1].scatter(H_xor[:, 0], H_xor[:, 1], c=colors, s=110, edgecolor="black")
axes[1].annotate("2 positivos", (1.0, 0.0), xytext=(0.62, 0.14), arrowprops={"arrowstyle": "->"})
h2_line = np.linspace(-0.1, 0.35, 50)
axes[1].plot(2 * h2_line + 0.5, h2_line, "k--", label="$h_1-2h_2-0{,}5=0$")
axes[1].set(title="Representação oculta: OR e AND", xlabel="$h_1$ (OR)", ylabel="$h_2$ (AND)")
axes[1].legend(loc="upper left")

for ax in axes:
    ax.set_xlim(-0.15, 1.2)
    ax.set_ylim(-0.15, 1.2)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.grid(alpha=0.25)
fig.suptitle("A camada oculta transforma o problema")
fig.tight_layout()
plt.show()


### 10. Sem não linearidade, as camadas colapsam

Duas transformações afins consecutivas são outra transformação afim:

$$
(XW^{[1]}+b^{[1]})W^{[2]}+b^{[2]}
=X(W^{[1]}W^{[2]})+(b^{[1]}W^{[2]}+b^{[2]}).
$$

O teste abaixo verifica a identidade com matrizes aleatórias compatíveis.


In [ ]:
X_linear = rng.normal(size=(13, 3))
W1_linear = rng.normal(size=(3, 5))
b1_linear = rng.normal(size=(1, 5))
W2_linear = rng.normal(size=(5, 2))
b2_linear = rng.normal(size=(1, 2))

stacked = dense(dense(X_linear, W1_linear, b1_linear), W2_linear, b2_linear)
W_collapsed = W1_linear @ W2_linear
b_collapsed = b1_linear @ W2_linear + b2_linear
collapsed = dense(X_linear, W_collapsed, b_collapsed)
collapse_error = float(np.max(np.abs(stacked - collapsed)))
assert collapse_error < 1e-12
print({"erro_maximo_colapso_afim": collapse_error})


## Checks

Esta célula consolida os contratos usados como evidência. Ela não substitui a execução
das células anteriores: depende deliberadamente de todos os resultados já produzidos.


In [ ]:
checks = {
    "exemplo_manual": np.array_equal(Z2_hand, expected_Z2),
    "matriz_equivale_neuronios": equivalence_error < 1e-14,
    "parametros_independentes_lote": formula_count == 26,
    "permutacao_lote": permutation_error < 1e-14,
    "independencia_lote": batch_independence_error < 1e-14,
    "linhas_duplicadas": duplicate_error < 1e-14,
    "broadcasting_detectado": wrong_contract_error > 0 and strict_rejected,
    "xor_exato": np.array_equal(y_hat_xor, y_xor),
    "camadas_afins_colapsam": collapse_error < 1e-12,
}
assert all(checks.values())
print(checks)
print(f"{sum(checks.values())}/{len(checks)} contratos satisfeitos")


## Limites do experimento

- Os pesos de XOR foram projetados manualmente; a rede ainda não aprendeu parâmetros.
- O degrau torna a lógica transparente, mas não será a escolha para backpropagation.
- Os testes cobrem shapes e simetrias do lote, não gradientes, estabilidade de treino ou
  generalização estatística.
- A convenção de exemplos em linhas não é universal. Outra convenção pode ser correta,
  desde que equações, código e documentação sejam internamente consistentes.


## Next

Na **Aula 04 — Funções de ativação**, vamos derivar sigmoid, tanh, ReLU e variantes,
comparar faixas e derivadas e explicar como a não linearidade impede o colapso afim.
Depois, a Aula 05 transformará este traço estrutural em um forward com cache adequado ao
backward manual.
